<style>
.rendered_html h1 { color: #1f4e79; }
.rendered_html h2 { color: #0f766e; }
.rendered_html h3, .rendered_html h4 { color: #7c3aed; }
</style>

# Data Preparation and Exploration

This notebook loads MovieLens 100K, explores the data, creates implicit feedback, builds train/test splits, and saves processed files for later notebooks.

## Setup

We import helper functions from `src/` so later notebooks can reuse the same preprocessing logic.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

from src.data_utils import (
    add_index_columns,
    compute_sparsity,
    create_implicit_feedback,
    create_mappings,
    ensure_project_dirs,
    leave_one_out_split,
    load_movielens_100k,
    save_processed_data,
)
from src.plotting_utils import (
    plot_interactions_per_movie,
    plot_interactions_per_user,
    plot_rating_distribution,
)

ensure_project_dirs(project_root)

data_dir = project_root / "dataset" / "ml-100k"
processed_dir = project_root / "data" / "processed"

## Load MovieLens 100K

We load ratings, users, and movies from the original dataset files.

In [ ]:
ratings, users, movies = load_movielens_100k(data_dir)

display(ratings.head())
display(users.head())
display(movies[["movie_id", "title", "release_date"]].head())

## Basic Exploration

We check shapes, missing values, and simple dataset counts.

In [ ]:
print("Ratings shape:", ratings.shape)
print("Users shape:", users.shape)
print("Movies shape:", movies.shape)

missing_values = pd.DataFrame({
    "ratings_missing": ratings.isna().sum(),
    "users_missing": users.isna().sum(),
    "movies_missing": movies.isna().sum(),
})

display(missing_values)

num_users = users["user_id"].nunique()
num_movies = movies["movie_id"].nunique()
num_ratings = len(ratings)

print("Number of users:", num_users)
print("Number of movies:", num_movies)
print("Number of ratings:", num_ratings)
print("Rating distribution:")
display(ratings["rating"].value_counts().sort_index())

## Initial Plots

These plots show how ratings and interactions are distributed.

In [ ]:
plot_rating_distribution(ratings)

## Explicit to Implicit Feedback

MovieLens is originally explicit feedback because users give ratings from 1 to 5. For an implicit recommendation system, we transform it into a liked/not-liked signal.

Here, `rating >= 4` means the user liked the movie, so `interaction = 1`.

In [ ]:
ratings_implicit, positive_interactions = create_implicit_feedback(ratings, threshold=4)

print("All ratings:", len(ratings_implicit))
print("Positive interactions:", len(positive_interactions))

display(positive_interactions.head())

## Positive Interaction Plots

We inspect how many positive interactions each user and movie has.

In [ ]:
plot_interactions_per_user(positive_interactions)
plot_interactions_per_movie(positive_interactions)

## Sparsity

Recommender datasets are sparse because most users interact with only a small number of items. Sparsity measures how much of the full user-item matrix is empty.

In [ ]:
sparsity = compute_sparsity(
    num_positive_interactions=len(positive_interactions),
    num_users=num_users,
    num_movies=num_movies,
)

print(f"Sparsity: {sparsity:.4%}")

## Leave-One-Out Split

For each user, the model learns from previous positive interactions and is tested on the latest liked item.

If the test set has 942 users instead of the original 943 MovieLens users, that is expected: after converting ratings to implicit positives with `rating >= 4`, some users may have no positive interaction left, so they cannot contribute a held-out positive test item.


In [ ]:
train_data, test_data = leave_one_out_split(positive_interactions)

print("Train interactions:", len(train_data))
print("Test interactions:", len(test_data))
print("Users in test data:", test_data["user_id"].nunique())

## Index Mappings

Models need continuous indexes starting from 0. We map original MovieLens ids to model-friendly indexes.

In [ ]:
user_to_idx, movie_to_idx, idx_to_user, idx_to_movie = create_mappings(users, movies)

train_data = add_index_columns(train_data, user_to_idx, movie_to_idx)
test_data = add_index_columns(test_data, user_to_idx, movie_to_idx)
users_processed = users.copy()
users_processed["user_idx"] = users_processed["user_id"].map(user_to_idx)
movies_processed = movies.copy()
movies_processed["movie_idx"] = movies_processed["movie_id"].map(movie_to_idx)

display(train_data[["user_id", "user_idx", "movie_id", "movie_idx", "interaction"]].head())

## Save Processed Data

We save the processed train/test splits, movies, users, and metadata so later notebooks can reuse them directly.

In [ ]:
metadata = {
    "user_to_idx": user_to_idx,
    "movie_to_idx": movie_to_idx,
    "idx_to_user": idx_to_user,
    "idx_to_movie": idx_to_movie,
    "num_users": num_users,
    "num_movies": num_movies,
    "sparsity": sparsity,
}

save_processed_data(
    train_data=train_data,
    test_data=test_data,
    movies=movies_processed,
    users=users_processed,
    metadata=metadata,
    output_dir=processed_dir,
)

print("Saved processed files to:", processed_dir)

## Conclusion

This notebook prepared MovieLens 100K for implicit recommendation. We created positive interactions, measured sparsity, built a leave-one-out split, created index mappings, and saved processed files for training and evaluation.

No model is trained in this notebook.